In [1]:
from google.colab import files

archivo = files.upload()

Saving practica_dml.db to practica_dml.db


In [2]:
import sqlite3

conexion = sqlite3.connect("practica_dml.db")
cursor = conexion.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
""")

print(cursor.fetchall())

[('vendedores',)]


In [3]:
# Crear la tabla productos
cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    stock INTEGER NOT NULL CHECK (stock >= 0)
)
""")

# Crear la tabla de auditoría
cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER,
    cambio_stock INTEGER,
    descripcion TEXT,
    fecha TEXT DEFAULT CURRENT_TIMESTAMP
)
""")

# Crear la tabla almacenes
cursor.execute("""
CREATE TABLE IF NOT EXISTS almacenes (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    stock INTEGER NOT NULL CHECK (stock >= 0)
)
""")

conexion.commit()

print("Las tres tablas fueron creadas correctamente.")

Las tres tablas fueron creadas correctamente.


In [4]:
# Cargar productos de prueba
cursor.executemany("""
INSERT INTO productos (id, nombre, stock)
VALUES (?, ?, ?)
""", [
    (1, 'Notebook', 20),
    (2, 'Mouse', 50),
    (3, 'Teclado', 30)
])

# Cargar almacenes de prueba
cursor.executemany("""
INSERT INTO almacenes (id, nombre, stock)
VALUES (?, ?, ?)
""", [
    (1, 'Almacén Central', 100),
    (2, 'Almacén Secundario', 50)
])

conexion.commit()

print("Datos de prueba cargados correctamente.")

Datos de prueba cargados correctamente.


In [5]:
# ACID: Consistencia. Verificamos el estado inicial antes de la transacción.

cursor.execute("""
SELECT * FROM productos
""")

for fila in cursor.fetchall():
    print(fila)

(1, 'Notebook', 20)
(2, 'Mouse', 50)
(3, 'Teclado', 30)


In [6]:
# CASO 1: TRANSACCIÓN EXITOSA
# ACID: Atomicidad y Durabilidad.
# El descuento de stock y el registro de auditoría
# se confirman juntos mediante COMMIT.

try:
    cursor.execute("BEGIN TRANSACTION")

    # Descontar 3 unidades de Notebook
    cursor.execute("""
    UPDATE productos
    SET stock = stock - 3
    WHERE id = 1
    """)

    # Registrar el movimiento en la auditoría
    cursor.execute("""
    INSERT INTO ventas_log (producto_id, cambio_stock, descripcion)
    VALUES (1, -3, 'Venta de 3 unidades de Notebook')
    """)

    conexion.commit()

    print("Transacción confirmada correctamente.")

except sqlite3.Error as error:
    conexion.rollback()
    print("Error:", error)
    print("Se deshicieron los cambios.")

Transacción confirmada correctamente.


In [7]:
# Verificar el stock actualizado
cursor.execute("""
SELECT id, nombre, stock
FROM productos
WHERE id = 1
""")

print("Stock actualizado:", cursor.fetchone())

# Verificar el registro de auditoría
cursor.execute("""
SELECT producto_id, cambio_stock, descripcion
FROM ventas_log
""")

print("Registro de auditoría:", cursor.fetchall())

Stock actualizado: (1, 'Notebook', 17)
Registro de auditoría: [(1, -3, 'Venta de 3 unidades de Notebook')]


In [8]:
# CASO 2: BOTÓN DE PÁNICO
# ACID: Atomicidad. Si la operación se cancela,
# los cambios no deben quedar aplicados.

# Mostrar los productos antes del intento de borrado
cursor.execute("SELECT * FROM productos")
print("Antes:", cursor.fetchall())

# Iniciar la transacción
cursor.execute("BEGIN TRANSACTION")

# Intentar borrar todos los productos
cursor.execute("DELETE FROM productos")

# Mostrar el resultado dentro de la transacción
cursor.execute("SELECT * FROM productos")
print("Durante la transacción:", cursor.fetchall())

# Cancelar el borrado
conexion.rollback()

# Comprobar que los productos siguen existiendo
cursor.execute("SELECT * FROM productos")
print("Después del ROLLBACK:", cursor.fetchall())

Antes: [(1, 'Notebook', 17), (2, 'Mouse', 50), (3, 'Teclado', 30)]
Durante la transacción: []
Después del ROLLBACK: [(1, 'Notebook', 17), (2, 'Mouse', 50), (3, 'Teclado', 30)]


In [9]:
# Verificar el stock inicial de los almacenes
cursor.execute("SELECT * FROM almacenes ORDER BY id")

for fila in cursor.fetchall():
    print(fila)

(1, 'Almacén Central', 100)
(2, 'Almacén Secundario', 50)


In [10]:
# CASO 3: ERROR DURANTE UNA TRANSFERENCIA
# ACID: Atomicidad. Si falla el segundo UPDATE,
# también debe deshacerse el primero.

cursor.execute("BEGIN TRANSACTION")

try:
    # Primer UPDATE: operación válida.
    cursor.execute("""
    UPDATE almacenes
    SET stock = stock - 10
    WHERE id = 1
    """)

    cursor.execute("SELECT * FROM almacenes ORDER BY id")
    print("Después del primer UPDATE:", cursor.fetchall())

    # Segundo UPDATE: provoca un error porque stock no admite NULL.
    cursor.execute("""
    UPDATE almacenes
    SET stock = NULL
    WHERE id = 2
    """)

    conexion.commit()
    print("Transacción confirmada.")

except sqlite3.IntegrityError as error:
    print("Error controlado:", error)

    # Deshacer también el primer UPDATE.
    conexion.rollback()
    print("Se ejecutó ROLLBACK.")

Después del primer UPDATE: [(1, 'Almacén Central', 90), (2, 'Almacén Secundario', 50)]
Error controlado: NOT NULL constraint failed: almacenes.stock
Se ejecutó ROLLBACK.


In [11]:
# Verificar el estado final después del ROLLBACK
cursor.execute("SELECT * FROM almacenes ORDER BY id")

print("Estado final de los almacenes:")

for fila in cursor.fetchall():
    print(fila)

Estado final de los almacenes:
(1, 'Almacén Central', 100)
(2, 'Almacén Secundario', 50)


In [12]:
contenido_sql = """
-- REPOSITORIO DE TRANSACCIONES SEGURAS EN SQLITE

-- Preparación de tablas
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    stock INTEGER NOT NULL CHECK (stock >= 0)
);

CREATE TABLE IF NOT EXISTS ventas_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER,
    cambio_stock INTEGER,
    descripcion TEXT,
    fecha TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS almacenes (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    stock INTEGER NOT NULL CHECK (stock >= 0)
);

-- CASO 1: TRANSACCIÓN EXITOSA
-- ACID: Atomicidad y Durabilidad.
-- El descuento y el registro de auditoría se confirman juntos.

-- Estado anterior: Notebook tiene 20 unidades.
BEGIN TRANSACTION;

UPDATE productos
SET stock = stock - 3
WHERE id = 1;

INSERT INTO ventas_log (producto_id, cambio_stock, descripcion)
VALUES (1, -3, 'Venta de 3 unidades de Notebook');

-- Verificación posterior al COMMIT:
COMMIT;

SELECT * FROM productos WHERE id = 1;
SELECT * FROM ventas_log;

-- CASO 2: BOTÓN DE PÁNICO
-- ACID: Atomicidad.
-- El borrado se cancela y los registros originales se conservan.

-- Verificación antes del DELETE:
SELECT * FROM productos;

BEGIN TRANSACTION;

DELETE FROM productos;

-- Estado temporal: tabla vacía dentro de la transacción.
SELECT * FROM productos;

ROLLBACK;

-- Verificación posterior al ROLLBACK:
SELECT * FROM productos;

-- CASO 3: ERROR DURANTE UNA TRANSFERENCIA
-- ACID: Atomicidad y Consistencia.
-- Si falla el segundo UPDATE, el primero también debe revertirse.

-- Estado inicial: Central 100 y Secundario 50.
SELECT * FROM almacenes;

BEGIN TRANSACTION;

-- Primer UPDATE: operación válida.
UPDATE almacenes
SET stock = stock - 10
WHERE id = 1;

-- Segundo UPDATE: genera un error por intentar asignar NULL
-- a una columna NOT NULL. Se debe ejecutar ROLLBACK.

UPDATE almacenes
SET stock = NULL
WHERE id = 2;

-- Ante el error, ejecutar ROLLBACK desde el programa
-- y comprobar que ambos stocks conservan sus valores iniciales.
-- SELECT * FROM almacenes;
"""

with open("transacciones_seguras.sql", "w", encoding="utf-8") as archivo:
    archivo.write(contenido_sql)

print("Archivo SQL creado correctamente.")

Archivo SQL creado correctamente.


In [13]:
from google.colab import files

files.download("transacciones_seguras.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>